#### **Step 1: Import Required Libraries**


In [6]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp)
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 8, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [7]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/kafka_eventstream/raw_stream/live_odds_updates"
df_raw_live_odds = spark.read.format("parquet").load(raw_transaction_path)

display(df_raw_live_odds.limit(10))

StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 257f562e-f43a-459a-bccd-fbff2497468b)

### **Column-by-Column Transformation Plan**
user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

update_id
- Current: Clean and consistent (e.g., 68fae11e-c393-44ec-a35e-7fc6d9351629).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the live_odds_update dataframe for data quality checks**

In [8]:
# Select specific columns and count nulls
null_counts = df_raw_live_odds.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['update_id', 'match_id', 'timestamp', 'market', 'odds_home', 'odds_draw', 'odds_away', 'event_date']
    ]
)

# Display results
null_counts.show()

StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 10, Finished, Available, Finished)

+---------+--------+---------+------+---------+---------+---------+
|update_id|match_id|timestamp|market|odds_home|odds_draw|odds_away|
+---------+--------+---------+------+---------+---------+---------+
|        0|       0|        0|     0|        0|        0|        0|
+---------+--------+---------+------+---------+---------+---------+



#### **Step 4: Data Transformation to outcome column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels (First Half, Handicap, Over/Under, 1X2).
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid outcomes (First Half, Handicap, Over/Under, 1X2) using a canonical character signature map.
- Returned standardized labels: First Half, Handicap, Over/Under, 1X2.
- Labeled unmatched or null values as "Unknown".

In [9]:
# 1. Pre‑compute canonical signatures
def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a word."""
    return "".join(sorted(word.lower()))

# Define the valid market values
canonical_values = ["First Half", "Handicap", "Over/Under", "1X2",]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)

# 2. UDF to normalize jumbled outcomes
@udf(StringType())
def normalize_outcome(raw: str) -> str:
    if raw is None:
        return None  # Return None to allow dropping later
    signature = "".join(sorted(raw.lower().strip()))
    return sig_broadcast.value.get(signature)


# 3. Apply transformation
df_cleaned_live_odds_update = df_raw_live_odds.withColumn(
    "market",
    normalize_outcome(col("market"))
)

# 4. Drop rows with unrecognized/null market values
df_cleaned_live_odds_update = df_cleaned_live_odds_update.filter(col("market").isNotNull())

# 5. Inspect cleaned market values
df_cleaned_live_odds_update.select("market").distinct().show(truncate=False)


StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 11, Finished, Available, Finished)

+----------+
|market    |
+----------+
|First Half|
|Handicap  |
|Over/Under|
|1X2       |
+----------+



#### **Step 5: Data Transformation to match_id column**
Issue:
- Some values had inconsistent casing like MATCH_867, match_432, or mixed-case variants.

Transformation:
- Converted all match_id values to lowercase.
- Trimmed any leading or trailing whitespace.

In [10]:
# Standardize `match_id` by converting to lowercase and trimming whitespace
df_cleaned_live_odds_update = df_cleaned_live_odds_update.withColumn(
    "match_id",
    lower(trim(col("match_id").cast("string")))
)

# Show sample records after transformation
df_cleaned_live_odds_update.select("match_id").show(10, truncate=False)


StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 12, Finished, Available, Finished)

+--------+
|match_id|
+--------+
|match_11|
|match_39|
|match_39|
|match_39|
|match_11|
|match_39|
|match_21|
|match_21|
|match_49|
|match_96|
+--------+
only showing top 10 rows



#### **Step 6: Data Transformation to timestamp column**
Issue:
- The column values were in ISO 8601 string format (yyyy-MM-ddTHH:mm:ssZ), not usable for date operations.

Transformation:
- Used to_timestamp() to convert string to proper timestamp format.
- Ensured time zone marker 'Z' was treated as a literal.

In [11]:
# Convert ISO 8601 format to timestamp
df_cleaned_live_odds_update = df_cleaned_live_odds_update.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Show sample converted values
df_cleaned_live_odds_update.select("timestamp").show(10, truncate=False)


StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 13, Finished, Available, Finished)

+-------------------+
|timestamp          |
+-------------------+
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
+-------------------+
only showing top 10 rows



#### **Step 7: Data Transformation to odds based columns**
Issue:
- Some odds values (odds_home, odds_draw, odds_away) were found to be negative, which is invalid in the context of sports betting odds.
- Negative odds could result from data entry errors or ingestion issues and may impact downstream analytics or user-facing features.

Transformation:
- Applied the absolute value function to each of the three odds columns to ensure all values are positive.
- This guarantees consistency and correctness in the representation of betting odds across the dataset.

In [12]:
# Apply absolute value transformation to odds columns
df_cleaned_live_odds_update = df_cleaned_live_odds_update.withColumn(
    "odds_home", abs(col("odds_home"))
).withColumn(
    "odds_draw", abs(col("odds_draw"))
).withColumn(
    "odds_away", abs(col("odds_away"))
)

display(df_cleaned_live_odds_update.show())

StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 14, Finished, Available, Finished)

+--------------------+--------+-------------------+----------+---------+---------+---------+----------+
|           update_id|match_id|          timestamp|    market|odds_home|odds_draw|odds_away|event_date|
+--------------------+--------+-------------------+----------+---------+---------+---------+----------+
|cf0eed57-1b28-4c0...|match_11|2025-07-05 08:35:19|Over/Under|      1.9|     2.92|     2.08|2025-07-05|
|23677ece-2385-4b6...|match_39|2025-07-05 08:35:19|First Half|     3.31|     3.59|     2.45|2025-07-05|
|16f45a7a-b4d0-41b...|match_39|2025-07-05 08:35:19|Over/Under|     1.39|     1.06|     2.08|2025-07-05|
|82901e86-c42e-4ff...|match_39|2025-07-05 08:35:19|  Handicap|     1.35|     2.12|     3.11|2025-07-05|
|06d98c46-7e7e-49a...|match_11|2025-07-05 08:35:19|First Half|     3.04|     1.83|     4.18|2025-07-05|
|36a8340f-3d9a-4af...|match_39|2025-07-05 08:35:19|       1X2|     2.15|     2.39|     3.49|2025-07-05|
|d3451b29-c3cb-4d2...|match_21|2025-07-05 08:35:19|  Handicap|  

#### **Step 9: Data Quality Checks**
- This step performs quality checks on critical columns of the df_cleaned_live_odds_update DataFrame to ensure the data is clean, accurate, and ready for downstream consumption.

| Check | Description |
| ---------------------- | ------------------------------------------------------------- |
| null_update_id | Must not be null |
| null_match_id | Must not be null |
| null_timestamp | Must not be null |
| null_market | Market (e.g., Handicap, 1X2) must be present |
| null_event_date | Event date must be present |
| negative_odds_home | Home odds must be non-negative |
| negative_odds_draw | Draw odds must be non-negative |
| negative_odds_away | Away odds must be non-negative |
| zero_or_low_odds | All odds should be greater than 0 (realistic threshold) |
| future_event_date | Event date must not be in the past relative to current date |

In [13]:
from pyspark.sql.functions import col, current_date, count, when

# --- Step 1: Filter out rows with non-positive odds ---
df_filtered = df_cleaned_live_odds_update.filter(
    (col("odds_home") > 0) &
    (col("odds_draw") > 0) &
    (col("odds_away") > 0)
)

# --- Step 2: Data Quality Check Metrics ---
dq_report = df_filtered.select(
    count("*").alias("total_rows"),
    count(when(col("update_id").isNull(), True)).alias("null_update_id"),
    count(when(col("match_id").isNull(), True)).alias("null_match_id"),
    count(when(col("timestamp").isNull(), True)).alias("null_timestamp"),
    count(when(col("market").isNull(), True)).alias("null_market"),

    count(when(col("odds_home") < 0, True)).alias("negative_odds_home"),
    count(when(col("odds_draw") < 0, True)).alias("negative_odds_draw"),
    count(when(col("odds_away") < 0, True)).alias("negative_odds_away"),

    count(when((col("odds_home") <= 0) | 
               (col("odds_draw") <= 0) |
               (col("odds_away") <= 0), True)).alias("zero_or_low_odds")
).collect()[0]

# --- Step 3: Print formatted DQ Report ---
print("LIVE ODDS DATA QUALITY REPORT")
print("--------------------------------------------------")
for k, v in dq_report.asDict().items():
    if k == "total_rows":
        status = "ℹ️ Row count only"
    elif k == "future_event_date":
        status = "✅ Expected (future match)"
    else:
        status = "✅ OK" if v == 0 else f"❌ {v} issue(s)"
    print(f"{k:<25}: {v:>5}  ->  {status}")
print("--------------------------------------------------")

StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 15, Finished, Available, Finished)

LIVE ODDS DATA QUALITY REPORT
--------------------------------------------------
total_rows               : 32257  ->  ℹ️ Row count only
null_update_id           :     0  ->  ✅ OK
null_match_id            :     0  ->  ✅ OK
null_timestamp           :     0  ->  ✅ OK
null_market              :     0  ->  ✅ OK
negative_odds_home       :     0  ->  ✅ OK
negative_odds_draw       :     0  ->  ✅ OK
negative_odds_away       :     0  ->  ✅ OK
zero_or_low_odds         :     0  ->  ✅ OK
--------------------------------------------------


#### **Step 10: Data Load to Silver Lakehouse Layer**
This step writes the cleaned users data to the Silver layer of the Lakehouse using Delta format. It uses a MERGE strategy to enable incremental upserts—updating existing user preferences and inserting new ones—while avoiding duplicates.

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/kafka_eventstream/cleaned/cleaned_live_odds_update.

Prepare Columns for Consistency:
- event_date: Adds the current date for use as a partition column to optimize query performance and simplify incremental loads.
- load_timestamp: Adds a consistent timestamp string (formatted as YYYYMMDDHHMMSS) that represents when the data was loaded.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key user_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by ingestion_date for efficient querying and file organization.


In [14]:
# Define paths
silver_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_live_odds_update"

# Add date column for partitioning (based on event_date if available)
df_cleaned_live_odds_update = df_cleaned_live_odds_update.withColumn(
    "event_date", to_date("event_date")  
)

# Add a consistent load timestamp column (once per job run)
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_live_odds_update = df_cleaned_live_odds_update.withColumn(
    "load_timestamp", lit(load_ts)
)

# Perform MERGE if Delta table exists
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target").merge(
        df_cleaned_live_odds_update.alias("source"),
        "target.update_id = source.update_id"  # Replace with your unique key
    ).whenNotMatchedInsertAll().execute()

else:
    # First write: create partitioned Delta table
    df_cleaned_live_odds_update.write.format("delta") \
        .mode("append") \
        .partitionBy("timestamp") \
        .save(silver_path)


StatementMeta(, f272660e-0b37-42a1-952b-718e51e21521, 16, Finished, Available, Finished)